In [5]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("valhalla/bart-large-sst2")
model = AutoModelForSequenceClassification.from_pretrained("valhalla/bart-large-sst2")

/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
/home/ubuntu/anaconda3/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [2]:
print(model)

BartForSequenceClassification(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [6]:
#obfuscate model
model = model_obfuscation(model)
print(model)

Obfuscating: model.encoder.layers.0.self_attn.k_proj
Obfuscating: model.encoder.layers.0.self_attn.v_proj
Obfuscating: model.encoder.layers.0.self_attn.q_proj
Obfuscating: model.encoder.layers.0.self_attn.out_proj
Obfuscating: model.encoder.layers.0.fc1
Obfuscating: model.encoder.layers.0.fc2
Obfuscating: model.encoder.layers.1.self_attn.k_proj
Obfuscating: model.encoder.layers.1.self_attn.v_proj
Obfuscating: model.encoder.layers.1.self_attn.q_proj
Obfuscating: model.encoder.layers.1.self_attn.out_proj
Obfuscating: model.encoder.layers.1.fc1
Obfuscating: model.encoder.layers.1.fc2
Obfuscating: model.encoder.layers.2.self_attn.k_proj
Obfuscating: model.encoder.layers.2.self_attn.v_proj
Obfuscating: model.encoder.layers.2.self_attn.q_proj
Obfuscating: model.encoder.layers.2.self_attn.out_proj
Obfuscating: model.encoder.layers.2.fc1
Obfuscating: model.encoder.layers.2.fc2
Obfuscating: model.encoder.layers.3.self_attn.k_proj
Obfuscating: model.encoder.layers.3.self_attn.v_proj
Obfuscating:

In [7]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [8]:
# two examples from training set

sentence = "contains no wit , only labored gags"

print(classifier(sentence))

sentence = "are more deeply thought through than in most ` right-thinking ' films"
print(classifier(sentence))

[{'label': 'NEGATIVE', 'score': 0.999679684638977}]
[{'label': 'POSITIVE', 'score': 0.9998600482940674}]


In [9]:
dataset = load_dataset("glue", "sst2")['validation']
print(dataset)

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 872
})


In [10]:
correct = 0
total = 0

for i in tqdm.tqdm(range(872)):
    result = classifier(dataset[i]['sentence'])
    result = 0 if result[0]['label'] == 'NEGATIVE' else 1
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 872/872 [00:23<00:00, 37.33it/s]

correct:831, total:872, accuracy:0.9529816513761468
